# Neural MCE-IRL applied workflow

This notebook fits an anchored neural reward, inspects the learned policy, requests whole-trajectory bootstrap intervals, runs a reward counterfactual, and verifies serialization. The transition tensor uses `(actions, states, next_states)` orientation.

In [1]:
import pickle
from pathlib import Path

import jax.numpy as jnp
import numpy as np

import econirl
from econirl import MCEIRLNeural
from econirl.core.types import Panel, Trajectory

working_directory = Path.cwd().resolve()
checkout_root = next(
    (path for path in (working_directory, *working_directory.parents) if (path / "src/econirl").is_dir()),
    None,
)
module_path = Path(econirl.__file__).resolve()
module_outside_checkout = checkout_root is None or checkout_root not in module_path.parents
print(f"Installed package import: {module_outside_checkout}")
print(f"Package version: {econirl.__version__}")

Installed package import: True
Package version: 0.0.10


In [2]:
transitions = np.zeros((2, 2, 2), dtype=np.float32)
transitions[:, 0, 0] = 1.0
transitions[:, 1, 1] = 1.0
def make_panel(seed, n_individuals):
    rng = np.random.default_rng(seed)
    trajectories = []
    for individual in range(n_individuals):
        state = individual % 2
        probability = 0.75 if state == 0 else 0.35
        action = int(rng.binomial(1, probability))
        trajectories.append(
            Trajectory(
                states=jnp.array([state]),
                actions=jnp.array([action]),
                next_states=jnp.array([state]),
                individual_id=individual,
            )
        )
    return Panel(trajectories)

panel = make_panel(2026, 120)
holdout = make_panel(2027, 60)
print(panel.num_individuals, panel.num_observations, transitions.shape)

120 120 (2, 2, 2)


In [3]:
model = MCEIRLNeural(
    n_states=2,
    n_actions=2,
    discount=0.9,
    reward_hidden_dim=8,
    reward_num_layers=1,
    max_epochs=150,
    lr=0.05,
    occupancy_tol=0.04,
    patience=50,
    anchor_action=0,
    se_method="bootstrap",
    n_bootstrap=3,
    se_seed=29,
    seed=4,
)
model.fit(panel, transitions=transitions)
print(model.summary())

Estimator
  MCEIRLNeural (Neural MCE-IRL)
Data
  Observations: 120
  States x actions: 2 x 2
Model
  Reward type: state_action
  Normalization: anchor_action=0
  Network: 1 hidden layer with 8 units
Pre-estimation checks
  State coverage: 1.000
  State-action coverage: 1.000
Fit
  Epochs: 103 (best 53)
  Converged: yes
  Termination: converged
  Fit time: 6.769 seconds
Outcome
  Demonstration log likelihood: -65.2489
  Occupancy residual: 0.00327464
  Bellman residual: 3.55271e-15
Uncertainty
  Method: whole-trajectory pairs bootstrap
  Bootstrap successful draws: 3/3
  Targets: anchored reward cells and policy probabilities
Limitations
  Network weights are not economic parameters.
  Reward levels depend on the stated normalization.
  Descriptive projection coordinates carry no sampling inference.


In [4]:
states = np.array([0, 1])
print(np.round(model.predict_proba(states), 4))
holdout_states = np.asarray(holdout.get_all_states(), dtype=int)
holdout_actions = np.asarray(holdout.get_all_actions(), dtype=int)
holdout_probabilities = model.predict_proba(holdout_states)
held_out_nll = -np.log(holdout_probabilities[np.arange(len(holdout_actions)), holdout_actions]).mean()
print(f"Held-out negative log likelihood: {held_out_nll:.4f}")
print(model.diagnostics_)
print({name: tuple(round(x, 4) for x in bounds) for name, bounds in model.conf_int().items()})

[[0.1768 0.8232]
 [0.705  0.295 ]]
Held-out negative log likelihood: 0.6078
{'data': {'n_observations': 120, 'n_individuals': 120, 'n_states_declared': 2, 'n_states_observed': 2, 'n_actions_declared': 2, 'state_coverage': 1.0, 'state_action_coverage': 1.0, 'single_action_states': 0, 'min_action_share': 0.44166666666666665}, 'identification': {'target': 'anchored neural reward matrix and induced policy', 'normalization': 'anchor_action=0', 'feature_rank': None, 'feature_condition_number': None, 'contrast_rank': None, 'contrast_condition_number': None, 'effective_occupancy_support': 1.0, 'verdict': 'covered'}, 'transitions': {'source': 'supplied', 'orientation': '(n_actions, n_states, n_states)', 'shape': (2, 2, 2), 'finite': True, 'nonnegative': True, 'max_row_sum_error': 0.0}, 'optimization': {'converged': True, 'termination_reason': 'converged', 'failure_reason': None, 'iterations': 103, 'fit_time_seconds': 6.768614917062223, 'occupancy_moment_residual': 0.0032746391741057534, 'bellma

In [5]:
reward_delta = np.zeros((2, 2))
reward_delta[:, 1] = 0.2
counterfactual = model.counterfactual(reward_delta=reward_delta)
print(np.round(counterfactual.counterfactual_policy, 4))
print(counterfactual.metadata["bootstrap_intervals"])

[[0.1495 0.8505]
 [0.6618 0.3382]]
{'method': 'pairs_cluster', 'n_successful': 3, 'mean_policy_tv': (0.032985726588107125, 0.03886729502754073), 'mean_value_change': (1.0857137077244101, 1.1888976175750965)}


In [6]:
restored = pickle.loads(pickle.dumps(model))
print(np.array_equal(restored.predict_proba(states), model.predict_proba(states)))
print(restored.econirl_version_)

True
0.0.10
